In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re, torch
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/save_hiddens'
out_dir = f'{data_dir}/visualize_hiddens'

In [ ]:
# def visualize_pca_2d_png(layer_idx, X_pca, y, out_prefix):
#     plt.figure(figsize=(8, 6))
#     plt.scatter(
#         X_pca[y == 'Fact', 0], # x축 좌표: Fact 그룹의 PC1 값들
#         X_pca[y == 'Fact', 1], # y축 좌표: Fact 그룹의 PC2 값들
#         alpha=0.6,             # 투명도: 0.6 (점이 겹칠 때 진해지도록 해서 밀집도를 보기 위함)
#         label='Fact',          # 범례(Legend)에 표시될 이름
#         c='blue',              # 점의 색상: 파란색
#         s=15                   # 점의 크기: 15
#     )
#     plt.scatter(X_pca[y == 'Counter', 0], X_pca[y == 'Counter', 1], alpha=0.6, label='Counter', c='red', s=15)
#     plt.title(f'PCA Visualization - Layer {layer_idx}')
#     plt.xlabel('PC1')
#     plt.ylabel('PC2')
#     plt.legend()
#     plt.grid(True, linestyle='--', alpha=0.5)

#     '''
#         dpi=300: 해상도(Dots Per Inch)를 설정
#             - 기본값(보통 100)으로 저장하면 논문에 넣었을 때 글씨가 깨질 수 있으므로, 학술 규격인 300 이상으로 설정

#         bbox_inches='tight': 그래프 주변의 쓸데없는 흰색 여백(Margin)을 자동으로 잘라내어 축, 제목, 범례가 잘리지 않고 딱 맞게 캡처
#     '''
#     out_file_path = f'{out_prefix}/pca_layer_{layer_idx}.png'
#     file_utils.make_parent(out_file_path)
#     plt.savefig(out_file_path, dpi=300, bbox_inches='tight')

#     # 저장하고 show() 해야 함. show() 하면 내용이 지워짐
#     plt.show()

In [ ]:
# def visualize_pca_2d_html(layer_idx, df, explained_var, out_prefix):
#     fig = px.scatter(
#         df, x='PC1', y='PC2', color='Group',
#         color_discrete_map={'Fact': 'blue', 'Counter': 'red'},
#         opacity=0.7,
#         title=f'PCA 2D - Layer {layer_idx} (Top 2 Var: {sum(explained_var[:2]):.2f}%)',
#         labels={
#             'PC1': f'PC1 ({explained_var[0]:.2f}%)',
#             'PC2': f'PC2 ({explained_var[1]:.2f}%)'
#         },
#         width=800,
#         height=600
#     )

#     out_file_path = f'{out_prefix}/pca_2d_layer_{layer_idx}.html'
#     file_utils.make_parent(out_file_path)
#     fig.write_html(f'{out_file_path}.html')

#     fig.show()

In [ ]:
def get_file_path_pca(out_prefix, n_comp, file_type, layer_idx):
    out_file_path = f'{out_prefix}/pca_{n_comp}d/{file_type}/pca_{n_comp}d_layer_{layer_idx}.{file_type}'
    file_utils.make_parent(out_file_path)

    return out_file_path

In [ ]:
def visualize_pca_2d(layer_idx, df, explained_var, out_prefix):
    # 1. 상세 확인용 인터랙티브 HTML (Plotly)
    fig_html = px.scatter(
        df, x='PC1', y='PC2', color='Group',
        color_discrete_map={'Fact': 'blue', 'Counter': 'red'},
        opacity=0.7,
        title=f'PCA 2D - Layer {layer_idx} (Top 2 Var: {sum(explained_var[:2]):.2f}%)',
        labels={
            'PC1': f'PC1 ({explained_var[0]:.2f}%)',
            'PC2': f'PC2 ({explained_var[1]:.2f}%)'
        },
        width=800,
        height=600
    )
    fig_html.write_html(get_file_path_pca(out_prefix, 2, 'html', layer_idx))

    # 2. 썸네일 스캔용 정적 PNG (Matplotlib)
    plt.figure(figsize=(8, 6))

    fact_data = df[df['Group'] == 'Fact']
    counter_data = df[df['Group'] == 'Counter']

    plt.scatter(fact_data['PC1'], fact_data['PC2'], alpha=0.6, label='Fact', c='blue', s=15)
    plt.scatter(counter_data['PC1'], counter_data['PC2'], alpha=0.6, label='Counter', c='red', s=15)
    
    plt.title(f'PCA 2D - Layer {layer_idx} (Top 2 Var: {sum(explained_var[:2]):.2f}%)')
    plt.xlabel(f'PC1 ({explained_var[0]:.2f}%)')
    plt.ylabel(f'PC2 ({explained_var[1]:.2f}%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.savefig(get_file_path_pca(out_prefix, 2, 'png', layer_idx), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close() # 메모리 누수 방지

In [ ]:
def visualize_pca_3d(layer_idx, df, explained_var, out_prefix):
    out_file_path = f'{out_prefix}/pca_3d_layer_{layer_idx}'
    file_utils.make_parent(out_file_path)

    # 1. 상세 확인용 인터랙티브 HTML (Plotly)
    fig_html = px.scatter_3d(
        df, x='PC1', y='PC2', z='PC3', color='Group',
        color_discrete_map={'Fact': 'blue', 'Counter': 'red'},
        opacity=0.7,
        title=f'PCA 3D - Layer {layer_idx} (Top 3 Var: {sum(explained_var[:3]):.2f}%)',
        labels={
            'PC1': f'PC1 ({explained_var[0]:.2f}%)',
            'PC2': f'PC2 ({explained_var[1]:.2f}%)',
            'PC3': f'PC3 ({explained_var[2]:.2f}%)'
        },
        width=800,
        height=600
    )
    fig_html.update_traces(marker=dict(size=3))
    fig_html.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig_html.write_html(get_file_path_pca(out_prefix, 3, 'html', layer_idx))

    # 2. 썸네일 스캔용 정적 PNG (Matplotlib)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d') # 3D 축 생성

    fact_data = df[df['Group'] == 'Fact']
    counter_data = df[df['Group'] == 'Counter']

    ax.scatter(fact_data['PC1'], fact_data['PC2'], fact_data['PC3'], alpha=0.6, label='Fact', c='blue', s=15)
    ax.scatter(counter_data['PC1'], counter_data['PC2'], counter_data['PC3'], alpha=0.6, label='Counter', c='red', s=15)
    
    ax.set_title(f'PCA 3D - Layer {layer_idx} (Top 3 Var: {sum(explained_var[:3]):.2f}%)')
    ax.set_xlabel(f'PC1 ({explained_var[0]:.2f}%)')
    ax.set_ylabel(f'PC2 ({explained_var[1]:.2f}%)')
    ax.set_zlabel(f'PC3 ({explained_var[2]:.2f}%)')
    ax.legend()
    
    plt.savefig(get_file_path_pca(out_prefix, 3, 'png', layer_idx), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close() # 메모리 누수 방지

In [ ]:
def visualize_pca_by_layer(model_name, num_layers, in_dir, out_dir, n_components=10):
    sum_explained_vars = {}

    '''
        - Layer '0'은 단순 단어 사전
            - Layer 0(hidden_states[0])은 트랜스포머의 어텐션(Attention) 블록을 타기 전의 가장 날것의 단어 임베딩(Word Embedding)
            - 프롬프트를 Question: {query}, Context: {context}, Answer: 형태로 만들었기 때문에 모든 마지막 토큰이 같음
            - 모든 데이터의 마지막 토큰이 동일하므로, Layer 0에서는 모든 데이터가 소수점 끝자리까지 완벽하게 똑같은 벡터
    '''
    for layer_idx in range(1, num_layers+1):
        # 1. 데이터 로드
        fact_file_path = f'{in_dir}/{model_name}/fact/layer_{layer_idx}_h_states.pt'
        counter_file_path = f'{in_dir}/{model_name}/counter/layer_{layer_idx}_h_states.pt'

        layer_h_states_fact = torch.load(fact_file_path, weights_only=True).numpy()
        layer_h_states_counter = torch.load(counter_file_path, weights_only=True).numpy()
        print(f'layer_h_states_fact shape : {layer_h_states_fact.shape}')
        print(f'layer_h_states_counter shape : {layer_h_states_counter.shape}')

        # 2. 데이터 결합 및 라벨링
        X = np.vstack([layer_h_states_fact, layer_h_states_counter])
        y = np.array(['Fact']*len(layer_h_states_fact) + ['Counter']*len(layer_h_states_counter))
        print(f'X shape : {X.shape}')
        print(f'y shape : {y.shape}')

        # 3. 데이터 스케일링 (PCA 전 필수 과정)
        X_scaled = StandardScaler().fit_transform(X)

        # 4. PCA 수행
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        print(f'X_pca shape : {X_pca.shape}\n')

        # 설명 분산 비율 출력 (데이터가 얼마나 잘 표현되었는지 확인)
        explained_var = pca.explained_variance_ratio_ * 100
        for i in range(n_components):
            print(f'\tPC{i+1} : {explained_var[i]:.4f}%')
        print()

        sum_explained_var = sum(explained_var[:3])
        sum_explained_vars[f'layer_{layer_idx}'] = sum_explained_var
        print(f'layer_{layer_idx} - PCA Explained Variance (Top 3 Sum) : {sum_explained_var:.4f}%')

        # 시각화를 위한 DF 생성
        df = pd.DataFrame({'Group': y})
        for i in range(3):
            df[f'PC{i+1}'] = X_pca[:, i]
        
        # 5. 시각화
        visualize_pca_2d(layer_idx, df, explained_var, f'{out_dir}/{model_name}')
        visualize_pca_3d(layer_idx, df, explained_var, f'{out_dir}/{model_name}')
    
    sorted_sum_explained_vars = container_utils.sorted_dict_value(sum_explained_vars)
    for key in sorted_sum_explained_vars.keys():
        print(f'{key} - PCA Explained Variance (sum) : {sorted_sum_explained_vars[key]:.4f}')

In [ ]:
visualize_pca_by_layer('Llama-3.2-3B', 28, in_dir, out_dir)

In [ ]:
visualize_pca_by_layer('Llama-3.1-8B', 32, in_dir, out_dir)